In [1]:
import argparse
import os
import pickle
import time

from importlib import metadata
import torch
try:
    try:
        if metadata.version("rsl-rl"):
            raise ImportError
    except metadata.PackageNotFoundError:
        if metadata.version("rsl-rl-lib") != "3.1.1":  #2.2.4
            raise ImportError
except (metadata.PackageNotFoundError, ImportError) as e:
    raise ImportError("Please uninstall 'rsl_rl' and install 'rsl-rl-lib==2.2.4'.") from e
from rsl_rl.runners import OnPolicyRunner

In [2]:
from bp000_env_cnoid import BP000Env as RLEnv

In [3]:
# 任意設定項目
# exp_name = 'friction-walking-terrain2-kp2000kd50-kpkdrand-ridho-model'  # ckpt = 4000
# exp_name = 'friction-walking-terrain1-kp2000kd50-kpkdrand-26-norand'
# exp_name = 'friction-walking-terrain2-kp2000kd50-kpkdrand-9'  # min_ankle_height 弊害
exp_name = 'friction-walking-terrain2-kp2000kd50-kpkdrand-8'  #  暫定１位
# exp_name = 'bp000-walking'
# exp_name = 'friction-walking-fractal-norand'
ckpt = 100

action_scale = 1.0 # 動作のスケールを調整

In [4]:
# 既存のセルを置き換え
import pandas as pd
import numpy as np

# データ収集用のリスト
# action_data = []
obs_data = []
torque_data = []
step_data = []
dof_pos_data = []
dof_vel_data = []

# CSVファイルの準備
csv_filename = f'obs_data/{exp_name}_step_data.csv'
os.makedirs('obs_data', exist_ok=True)

In [5]:
def _obs_vec(obs):
    # TensorDict or dict → 'policy' を優先
    if isinstance(obs, dict) or hasattr(obs, "get"):
        if "policy" in obs:
            obs = obs["policy"]
    if torch.is_tensor(obs):
        return obs.detach().cpu().numpy().ravel()
    return np.asarray(obs, dtype=np.float32).ravel()

In [6]:
## set robot path fix collisiton 
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))  # /userdir
robot_path = os.path.join(ROOT, "userdir", "humanoid_research_k", "robots", "kawada_base.simple_collision.urdf")

In [7]:
log_dir = f"logs/{exp_name}"
env_cfg, obs_cfg, reward_cfg, command_cfg, train_cfg = pickle.load(open(f"logs/{exp_name}/cfgs.pkl", "rb"))
reward_cfg["reward_scales"] = {}

In [8]:
## override
env_cfg["episode_length_s"] = 60.0
# command_cfg["lin_vel_x_range"] = [0.5, 0.5]
env_cfg['dt'] = 0.01
env_cfg['substeps'] = 10
# env_cfg["kd"] = 50
env_cfg['base_roll_noise'] = [0,0]
env_cfg['base_pitch_noise'] = [0,0]
env_cfg['termination_if_roll_greater_than'] = 150
env_cfg['termination_if_pitch_greater_than'] = 150
env_cfg['rotorInertia'] = 0.1
env_cfg["base_init_pos"] = [0.0, 0.0, 0.64]

In [9]:
reward_cfg

{'tracking_sigma': 0.25,
 'base_height_target': 0.64,
 'feet_height_target': 0.075,
 'reward_scales': {}}

In [10]:
env_cfg



{'num_actions': 12,
 'default_joint_angles': {'R_HIP_Y': 0.0,
  'R_HIP_R': 0.0,
  'R_HIP_P': -0.8,
  'R_KNEE': 1.6,
  'R_ANKLE_P': -0.8,
  'R_ANKLE_R': 0.0,
  'L_HIP_Y': 0.0,
  'L_HIP_R': 0.0,
  'L_HIP_P': -0.8,
  'L_KNEE': 1.6,
  'L_ANKLE_P': -0.8,
  'L_ANKLE_R': 0.0},
 'joint_names': ['R_HIP_Y',
  'R_HIP_R',
  'R_HIP_P',
  'R_KNEE',
  'R_ANKLE_P',
  'R_ANKLE_R',
  'L_HIP_Y',
  'L_HIP_R',
  'L_HIP_P',
  'L_KNEE',
  'L_ANKLE_P',
  'L_ANKLE_R'],
 'kp': 2000.0,
 'kd': 50.0,
 'termination_if_roll_greater_than': 150,
 'termination_if_pitch_greater_than': 150,
 'base_init_pos': [0.0, 0.0, 0.64],
 'base_init_quat': [1.0, 0.0, 0.0, 0.0],
 'episode_length_s': 60.0,
 'resampling_time_s': 4.0,
 'action_scale': 1.0,
 'simulate_action_latency': True,
 'clip_actions': 100.0,
 'dt': 0.01,
 'substeps': 10,
 'rotorInertia': 0.1,
 'base_roll_noise': [0, 0],
 'base_pitch_noise': [0, 0],
 'domain_rand': {'friction': [0.4, 1.1],
  'restitution': [0.0, 0.2],
  'kp': [1800.0, 2200.0],
  'kd': [25.0, 100.0]}

In [11]:
env = RLEnv(
    num_envs=1,
    env_cfg=env_cfg,
    obs_cfg=obs_cfg,
    reward_cfg=reward_cfg,
    command_cfg=command_cfg,
    dt=env_cfg['dt'],
    substeps=env_cfg['substeps'],
    show_viewer=True,
    robot_urdf_path=robot_path,
)

In [12]:
runner = OnPolicyRunner(env, train_cfg, log_dir, device='cuda')
resume_path = os.path.join(log_dir, f"model_{ckpt}.pt")
runner.load(resume_path)
policy = runner.get_inference_policy(device='cuda')

obs, _ = env.reset()
cnt = 0

torques = env.sim.sbody.getTorques()
dof_pos = env.dof_pos[0].cpu().numpy()
dof_vel = env.dof_vel[0].cpu().numpy()

print("obs : ", obs["policy"])

# データを記録
step_data.append(cnt)
obs_data.append(_obs_vec(obs))
torque_data.append(torques.copy())
dof_pos_data.append(dof_pos)
dof_vel_data.append(dof_vel)

cnt += 1

--------------------------------------------------------------------------------
Resolved observation sets: 
	 policy :  ['policy']
	 critic :  ['policy']
--------------------------------------------------------------------------------
Actor MLP: MLP(
  (0): Linear(in_features=45, out_features=512, bias=True)
  (1): ELU(alpha=1.0)
  (2): Linear(in_features=512, out_features=256, bias=True)
  (3): ELU(alpha=1.0)
  (4): Linear(in_features=256, out_features=128, bias=True)
  (5): ELU(alpha=1.0)
  (6): Linear(in_features=128, out_features=12, bias=True)
)
Critic MLP: MLP(
  (0): Linear(in_features=45, out_features=512, bias=True)
  (1): ELU(alpha=1.0)
  (2): Linear(in_features=512, out_features=256, bias=True)
  (3): ELU(alpha=1.0)
  (4): Linear(in_features=256, out_features=128, bias=True)
  (5): ELU(alpha=1.0)
  (6): Linear(in_features=128, out_features=1, bias=True)
)
obs :  tensor([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        

In [13]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    dof_pos = env.dof_pos[0].cpu().numpy()
    dof_vel = env.dof_vel[0].cpu().numpy()
    print("torques:", torques)
    print("dof_pos:", dof_pos)
    print("dof_vel:", dof_vel)

    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    dof_pos_data.append(dof_pos)
    dof_vel_data.append(dof_vel)

    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 1
Original actions :  tensor([[-0.2214, -0.2589, -0.5345, -0.6375, -0.7908,  0.0285, -0.0553, -0.0345,
          0.9609, -0.6897, -0.9299, -0.3576]], device='cuda:0')
Scaled actions :  tensor([[-0.2214, -0.2589, -0.5345, -0.6375, -0.7908,  0.0285, -0.0553, -0.0345,
          0.9609, -0.6897, -0.9299, -0.3576]], device='cuda:0')


/userdir/irsl_rl/rl_env_base.py:110: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.exact_actions = torch.tensor(actions, device=self.device, dtype=torch.float32) ## copy
/userdir/irsl_rl/rl_env_cnoid.py:103: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:254.)
  self.dof_pos = torch.tensor([self.convAnglesToGenesis(sbody.angleVector())]).to(torch.float32).to(self.device)


obs :  tensor([[-4.2367e-06, -7.9481e-03, -1.6194e-06,  5.0845e-10,  3.7697e-20,
         -1.0000e+00,  1.0000e+00,  0.0000e+00,  0.0000e+00, -8.5621e-08,
         -4.7636e-08, -1.4180e-04,  3.6466e-04, -1.9103e-04,  7.6633e-07,
          1.4362e-08,  3.0845e-08, -1.4168e-04,  3.6454e-04, -1.9103e-04,
         -3.4389e-08, -4.2811e-06, -2.3818e-06, -7.0895e-03,  1.8232e-02,
         -9.5524e-03,  3.8316e-05,  7.1809e-07,  1.5422e-06, -7.0859e-03,
          1.8227e-02, -9.5517e-03, -1.7194e-06, -2.2144e-01, -2.5885e-01,
         -5.3450e-01, -6.3750e-01, -7.9080e-01,  2.8542e-02, -5.5327e-02,
         -3.4531e-02,  9.6094e-01, -6.8968e-01, -9.2988e-01, -3.5762e-01]],
       device='cuda:0')
torques: [-1.58794924e-16 -1.00915112e-15  2.37314235e-06  7.56007923e-06
  1.59905156e-06  6.12617365e-17 -6.48048861e-18 -7.17891685e-16
  2.37314235e-06  7.56007923e-06  1.59905156e-06 -8.75046755e-17]
dof_pos: [-8.5621316e-08 -4.7636032e-08 -8.0014181e-01  1.6003647e+00
 -8.0019104e-01  7.6632642

In [13]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    dof_pos = env.dof_pos[0].cpu().numpy()
    dof_vel = env.dof_vel[0].cpu().numpy()
    print("torques:", torques)
    print("dof_pos:", dof_pos)
    print("dof_vel:", dof_vel)

    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    dof_pos_data.append(dof_pos)
    dof_vel_data.append(dof_vel)

    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 1
Original actions :  tensor([[-0.2214, -0.2589, -0.5345, -0.6375, -0.7908,  0.0285, -0.0553, -0.0345,
          0.9609, -0.6897, -0.9299, -0.3576]], device='cuda:0')
Scaled actions :  tensor([[-0.2214, -0.2589, -0.5345, -0.6375, -0.7908,  0.0285, -0.0553, -0.0345,
          0.9609, -0.6897, -0.9299, -0.3576]], device='cuda:0')


/userdir/irsl_rl/rl_env_base.py:110: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.exact_actions = torch.tensor(actions, device=self.device, dtype=torch.float32) ## copy
/userdir/irsl_rl/rl_env_cnoid.py:103: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:254.)
  self.dof_pos = torch.tensor([self.convAnglesToGenesis(sbody.angleVector())]).to(torch.float32).to(self.device)


obs :  tensor([[-4.2367e-06, -7.9481e-03, -1.6194e-06,  5.0845e-10,  3.7697e-20,
         -1.0000e+00,  1.0000e+00,  0.0000e+00,  0.0000e+00, -8.5621e-08,
         -4.7636e-08, -1.4180e-04,  3.6466e-04, -1.9103e-04,  7.6633e-07,
          1.4362e-08,  3.0845e-08, -1.4168e-04,  3.6454e-04, -1.9103e-04,
         -3.4389e-08, -4.2811e-06, -2.3818e-06, -7.0895e-03,  1.8232e-02,
         -9.5524e-03,  3.8316e-05,  7.1809e-07,  1.5422e-06, -7.0859e-03,
          1.8227e-02, -9.5517e-03, -1.7194e-06, -2.2144e-01, -2.5885e-01,
         -5.3450e-01, -6.3750e-01, -7.9080e-01,  2.8542e-02, -5.5327e-02,
         -3.4531e-02,  9.6094e-01, -6.8968e-01, -9.2988e-01, -3.5762e-01]],
       device='cuda:0')
torques: [-1.58794924e-16 -1.00915112e-15  2.37314235e-06  7.56007923e-06
  1.59905156e-06  6.12617365e-17 -6.48048861e-18 -7.17891685e-16
  2.37314235e-06  7.56007923e-06  1.59905156e-06 -8.75046755e-17]
dof_pos: [-8.5621316e-08 -4.7636032e-08 -8.0014181e-01  1.6003647e+00
 -8.0019104e-01  7.6632642

In [13]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    dof_pos = env.dof_pos[0].cpu().numpy()
    dof_vel = env.dof_vel[0].cpu().numpy()
    print("torques:", torques)
    print("dof_pos:", dof_pos)
    print("dof_vel:", dof_vel)

    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    dof_pos_data.append(dof_pos)
    dof_vel_data.append(dof_vel)

    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 1
Original actions :  tensor([[-0.2214, -0.2589, -0.5345, -0.6375, -0.7908,  0.0285, -0.0553, -0.0345,
          0.9609, -0.6897, -0.9299, -0.3576]], device='cuda:0')
Scaled actions :  tensor([[-0.2214, -0.2589, -0.5345, -0.6375, -0.7908,  0.0285, -0.0553, -0.0345,
          0.9609, -0.6897, -0.9299, -0.3576]], device='cuda:0')


/userdir/irsl_rl/rl_env_base.py:110: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.exact_actions = torch.tensor(actions, device=self.device, dtype=torch.float32) ## copy
/userdir/irsl_rl/rl_env_cnoid.py:103: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:254.)
  self.dof_pos = torch.tensor([self.convAnglesToGenesis(sbody.angleVector())]).to(torch.float32).to(self.device)


obs :  tensor([[-4.2367e-06, -7.9481e-03, -1.6194e-06,  5.0845e-10,  3.7697e-20,
         -1.0000e+00,  1.0000e+00,  0.0000e+00,  0.0000e+00, -8.5621e-08,
         -4.7636e-08, -1.4180e-04,  3.6466e-04, -1.9103e-04,  7.6633e-07,
          1.4362e-08,  3.0845e-08, -1.4168e-04,  3.6454e-04, -1.9103e-04,
         -3.4389e-08, -4.2811e-06, -2.3818e-06, -7.0895e-03,  1.8232e-02,
         -9.5524e-03,  3.8316e-05,  7.1809e-07,  1.5422e-06, -7.0859e-03,
          1.8227e-02, -9.5517e-03, -1.7194e-06, -2.2144e-01, -2.5885e-01,
         -5.3450e-01, -6.3750e-01, -7.9080e-01,  2.8542e-02, -5.5327e-02,
         -3.4531e-02,  9.6094e-01, -6.8968e-01, -9.2988e-01, -3.5762e-01]],
       device='cuda:0')
torques: [-1.58794924e-16 -1.00915112e-15  2.37314235e-06  7.56007923e-06
  1.59905156e-06  6.12617365e-17 -6.48048861e-18 -7.17891685e-16
  2.37314235e-06  7.56007923e-06  1.59905156e-06 -8.75046755e-17]
dof_pos: [-8.5621316e-08 -4.7636032e-08 -8.0014181e-01  1.6003647e+00
 -8.0019104e-01  7.6632642

In [13]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    dof_pos = env.dof_pos[0].cpu().numpy()
    dof_vel = env.dof_vel[0].cpu().numpy()
    print("torques:", torques)
    print("dof_pos:", dof_pos)
    print("dof_vel:", dof_vel)

    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    dof_pos_data.append(dof_pos)
    dof_vel_data.append(dof_vel)

    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 1
Original actions :  tensor([[-0.2214, -0.2589, -0.5345, -0.6375, -0.7908,  0.0285, -0.0553, -0.0345,
          0.9609, -0.6897, -0.9299, -0.3576]], device='cuda:0')
Scaled actions :  tensor([[-0.2214, -0.2589, -0.5345, -0.6375, -0.7908,  0.0285, -0.0553, -0.0345,
          0.9609, -0.6897, -0.9299, -0.3576]], device='cuda:0')


/userdir/irsl_rl/rl_env_base.py:110: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.exact_actions = torch.tensor(actions, device=self.device, dtype=torch.float32) ## copy
/userdir/irsl_rl/rl_env_cnoid.py:103: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:254.)
  self.dof_pos = torch.tensor([self.convAnglesToGenesis(sbody.angleVector())]).to(torch.float32).to(self.device)


obs :  tensor([[-4.2367e-06, -7.9481e-03, -1.6194e-06,  5.0845e-10,  3.7697e-20,
         -1.0000e+00,  1.0000e+00,  0.0000e+00,  0.0000e+00, -8.5621e-08,
         -4.7636e-08, -1.4180e-04,  3.6466e-04, -1.9103e-04,  7.6633e-07,
          1.4362e-08,  3.0845e-08, -1.4168e-04,  3.6454e-04, -1.9103e-04,
         -3.4389e-08, -4.2811e-06, -2.3818e-06, -7.0895e-03,  1.8232e-02,
         -9.5524e-03,  3.8316e-05,  7.1809e-07,  1.5422e-06, -7.0859e-03,
          1.8227e-02, -9.5517e-03, -1.7194e-06, -2.2144e-01, -2.5885e-01,
         -5.3450e-01, -6.3750e-01, -7.9080e-01,  2.8542e-02, -5.5327e-02,
         -3.4531e-02,  9.6094e-01, -6.8968e-01, -9.2988e-01, -3.5762e-01]],
       device='cuda:0')
torques: [-1.58794924e-16 -1.00915112e-15  2.37314235e-06  7.56007923e-06
  1.59905156e-06  6.12617365e-17 -6.48048861e-18 -7.17891685e-16
  2.37314235e-06  7.56007923e-06  1.59905156e-06 -8.75046755e-17]
dof_pos: [-8.5621316e-08 -4.7636032e-08 -8.0014181e-01  1.6003647e+00
 -8.0019104e-01  7.6632642

In [13]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    dof_pos = env.dof_pos[0].cpu().numpy()
    dof_vel = env.dof_vel[0].cpu().numpy()
    print("torques:", torques)
    print("dof_pos:", dof_pos)
    print("dof_vel:", dof_vel)

    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    dof_pos_data.append(dof_pos)
    dof_vel_data.append(dof_vel)

    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 1
Original actions :  tensor([[-0.2214, -0.2589, -0.5345, -0.6375, -0.7908,  0.0285, -0.0553, -0.0345,
          0.9609, -0.6897, -0.9299, -0.3576]], device='cuda:0')
Scaled actions :  tensor([[-0.2214, -0.2589, -0.5345, -0.6375, -0.7908,  0.0285, -0.0553, -0.0345,
          0.9609, -0.6897, -0.9299, -0.3576]], device='cuda:0')


/userdir/irsl_rl/rl_env_base.py:110: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.exact_actions = torch.tensor(actions, device=self.device, dtype=torch.float32) ## copy
/userdir/irsl_rl/rl_env_cnoid.py:103: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:254.)
  self.dof_pos = torch.tensor([self.convAnglesToGenesis(sbody.angleVector())]).to(torch.float32).to(self.device)


obs :  tensor([[-4.2367e-06, -7.9481e-03, -1.6194e-06,  5.0845e-10,  3.7697e-20,
         -1.0000e+00,  1.0000e+00,  0.0000e+00,  0.0000e+00, -8.5621e-08,
         -4.7636e-08, -1.4180e-04,  3.6466e-04, -1.9103e-04,  7.6633e-07,
          1.4362e-08,  3.0845e-08, -1.4168e-04,  3.6454e-04, -1.9103e-04,
         -3.4389e-08, -4.2811e-06, -2.3818e-06, -7.0895e-03,  1.8232e-02,
         -9.5524e-03,  3.8316e-05,  7.1809e-07,  1.5422e-06, -7.0859e-03,
          1.8227e-02, -9.5517e-03, -1.7194e-06, -2.2144e-01, -2.5885e-01,
         -5.3450e-01, -6.3750e-01, -7.9080e-01,  2.8542e-02, -5.5327e-02,
         -3.4531e-02,  9.6094e-01, -6.8968e-01, -9.2988e-01, -3.5762e-01]],
       device='cuda:0')
torques: [-1.58794924e-16 -1.00915112e-15  2.37314235e-06  7.56007923e-06
  1.59905156e-06  6.12617365e-17 -6.48048861e-18 -7.17891685e-16
  2.37314235e-06  7.56007923e-06  1.59905156e-06 -8.75046755e-17]
dof_pos: [-8.5621316e-08 -4.7636032e-08 -8.0014181e-01  1.6003647e+00
 -8.0019104e-01  7.6632642

In [17]:
# 既存のforループを置き換え
num_steps = 100
for i in range(num_steps):
    with torch.no_grad():
        actions = policy(obs)
        
        # アクションスケーリング
        scaled_actions = actions * action_scale
        obs, rews, dones, infos = env.step(scaled_actions)  # スケール済みを使用
        torques = env.sim.sbody.getTorques()
        dof_pos = env.dof_pos[0].cpu().numpy()
        dof_vel = env.dof_vel[0].cpu().numpy()
        
        # データを記録
        step_data.append(cnt)
        obs_data.append(_obs_vec(obs))
        torque_data.append(torques.copy())
        dof_pos_data.append(dof_pos)
        dof_vel_data.append(dof_vel)
        
        # デバッグ表示（最初の数ステップのみ）
        if i < 3:
            print(f"Step {i}: Original action max={actions.max():.3f}, "
                  f"Scaled action max={scaled_actions.max():.3f}")
        
        if i % 20 == 0:
            print(f"Step {i+1}/{num_steps}, Total steps: {cnt}")
            print("steps:",cnt)
            print("actions :",scaled_actions)
            print("target_dof_pos:",env.target_dof_pos)
        
        cnt += 1

print(f"データ収集完了: {num_steps} steps collected with action_scale={action_scale}")

Step 0: Original action max=1.235, Scaled action max=1.235
Step 1/100, Total steps: 401
steps: 401
actions : tensor([[ 0.4545, -1.1128, -0.8785, -1.0413, -0.9827,  0.5208,  1.2352, -1.1649,
         -0.4642, -0.2216, -0.5988, -0.7253]], device='cuda:0')
target_dof_pos: tensor([[-0.3476,  0.5283,  0.4879,  1.8687, -0.5557, -0.7933,  0.0573,  0.3770,
          0.0085,  1.9093, -0.4991,  0.8354]], device='cuda:0')
Step 1: Original action max=1.118, Scaled action max=1.118
Step 2: Original action max=1.236, Scaled action max=1.236
Step 21/100, Total steps: 421
steps: 421
actions : tensor([[ 0.6377,  1.0612,  0.5725,  0.3258,  2.1580,  0.6661, -0.1781,  0.9103,
         -0.1528,  0.0683, -0.3460,  1.1048]], device='cuda:0')
target_dof_pos: tensor([[-0.8413,  0.5319, -0.2070,  0.8171,  0.0846,  1.0079,  0.8156,  0.2066,
         -0.5835,  0.3811, -0.0523,  0.5914]], device='cuda:0')
Step 41/100, Total steps: 441
steps: 441
actions : tensor([[-0.8517,  0.6990,  0.7614, -0.5317, -0.9980, -1.21

In [18]:
# for i in range(500):
#     with torch.no_grad():
#         actions = policy(obs)
#         scaled_actions = actions * action_scale
#         obs, rews, dones, infos = env.step(scaled_actions)

In [19]:
env.sim.stop()

In [31]:
env.reset()
cnt = 0

In [20]:
# 最もシンプルな保存方法
def save_simple_csv():
    if not step_data:
        print("データがありません")
        return
    
    # 基本的な辞書形式でデータを整理
    data_dict = {'step': step_data}
    
    # # Actionデータ
    # action_array = np.array(action_data)
    # for i in range(action_array.shape[1]):
    #     data_dict[f'action_{i}'] = action_array[:, i]
    
    # Observationデータ
    obs_array = np.array(obs_data)
    for i in range(obs_array.shape[1]):
        data_dict[f'obs_{i}'] = obs_array[:, i]
    
    # Torqueデータ
    torque_array = np.array(torque_data)
    for i in range(torque_array.shape[1]):
        data_dict[f'torque_{i}'] = torque_array[:, i]

    # dof_posデータ
    dof_pos_array = np.array(dof_pos_data)
    for i in range(dof_pos_array.shape[1]):
        data_dict[f'dof_pos_{i}'] = dof_pos_array[:, i]

    # dof_velデータ
    dof_vel_array = np.array(dof_vel_data)
    for i in range(dof_vel_array.shape[1]):
        data_dict[f'dof_vel_{i}'] = dof_vel_array[:, i]
    
    # DataFrameを作成して保存
    df = pd.DataFrame(data_dict)
    csv_filename = f'obs_data/cnoid_{exp_name}_ckpt{ckpt}_scale{action_scale}_rotorInertia0.1.csv'
    df.to_csv(csv_filename, index=False)
    
    print(f"シンプル版を保存: {csv_filename}")
    print(f"データ形状: {df.shape}")
    
    return df

# シンプル版を実行
df_simple = save_simple_csv()

シンプル版を保存: obs_data/cnoid_friction-walking-terrain2-kp2000kd50-kpkdrand-8_ckpt100_scale1.0_rotorInertia0.1.csv
データ形状: (501, 82)
